# WING Fashion — Flux 產品攝影（ComfyUI 版）
㩒 **Runtime → Run all**，等 ~5分鐘自動出 7 張圖。
唔需要 HuggingFace token。

In [ ]:
# @title 1. Install ComfyUI + Flux
import os, sys, json, time, requests, threading, random
from pathlib import Path
from IPython.display import display, Image as IPImage

COMFY = "/content/ComfyUI"
os.makedirs(COMFY, exist_ok=True)

print("⚙️ Installing ComfyUI...")
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFY} 2>/dev/null
!cd {COMFY} && pip install -q -r requirements.txt 2>&1 | tail -1

# Download Flux fp8 model (from ComfyOrg, no auth needed)
model_dir = f"{COMFY}/models/checkpoints"
os.makedirs(model_dir, exist_ok=True)

if not os.path.exists(f"{model_dir}/flux1-dev-fp8.safetensors"):
    print("📥 Downloading Flux Dev fp8 (~12GB, 3-5 min)...")
    !wget -q --show-progress -O {model_dir}/flux1-dev-fp8.safetensors \
      https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors

# CLIP
clip_dir = f"{COMFY}/models/clip"
os.makedirs(f"{clip_dir}/t5xxl", exist_ok=True)
os.makedirs(f"{clip_dir}/clip_l", exist_ok=True)

if not os.path.exists(f"{clip_dir}/t5xxl_fp8.safetensors"):
    print("📥 Downloading T5 encoder...")
    !wget -q --show-progress -O {clip_dir}/t5xxl_fp8.safetensors \
      https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/t5xxl_fp8.safetensors

if not os.path.exists(f"{clip_dir}/clip_l.safetensors"):
    print("📥 Downloading CLIP-L...")
    !wget -q --show-progress -O {clip_dir}/clip_l.safetensors \
      https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/clip_l.safetensors

# VAE
if not os.path.exists(f"{COMFY}/models/vae/ae.safetensors"):
    os.makedirs(f"{COMFY}/models/vae", exist_ok=True)
    print("📥 Downloading VAE...")
    !wget -q --show-progress -O {COMFY}/models/vae/ae.safetensors \
      https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/ae.safetensors

print("✅ ComfyUI + Flux ready!")

In [ ]:
# @title 2. Start ComfyUI Server
def start_server():
    !cd {COMFY} && python main.py --dont-print-server --listen 0.0.0.0 --port 8188 2>&1

threading.Thread(target=start_server, daemon=True).start()

for i in range(60):
    try:
        r = requests.get("http://127.0.0.1:8188/system_stats", timeout=2)
        if r.status_code == 200:
            print("✅ Server running!")
            break
    except: pass
    time.sleep(2)
else:
    print("❌ Server failed")
    raise SystemExit()

In [ ]:
# @title 3. Generate 7 Photos
import requests, json, base64, io

STYLE = "editorial fashion photography, cold grey color palette, soft natural lighting, cinematic composition, clean minimalist background, 1:1 aspect ratio, high end luxury look, muted tones"

prompts = [
  "a Fred Perry diamond-print shirt hung on a minimalist metal rack, studio lighting, soft shadows, cold grey concrete wall background, " + STYLE,
  "a Fred Perry diamond-print shirt neatly folded on a dark grey marble surface, top-down view, soft diffused light, " + STYLE,
  "a Fred Perry diamond-print shirt on a minimalist mannequin torso, three-quarter angle, grey seamless backdrop, " + STYLE,
  "close up detail of a Fred Perry diamond-print shirt fabric texture, macro photography, soft light, grey background, " + STYLE,
  "Minimalist interior corner, white walls, grey concrete floor, soft window light, geometric shadows, empty, editorial architectural photography, " + STYLE,
  "Abstract grey gradient background, subtle texture, smooth light to dark grey, minimal fashion editorial backdrop, " + STYLE,
  "Empty minimalist room, natural light through sheer curtains, pale grey walls, subtle shadows, calm sophisticated atmosphere, " + STYLE,
]

os.makedirs("/content/wing_output", exist_ok=True)
results = []

for idx, prompt in enumerate(prompts):
    print(f"\n🎨 Generating {idx+1}/7...")
    
    wf = {
        "3": {"class_type": "KSampler", "inputs": {"seed": random.randint(1,999999), "steps": 30, "cfg": 1.0, "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0, "model": ["10",0], "positive": ["6",0], "negative": ["7",0], "latent_image": ["5",0]}},
        "5": {"class_type": "EmptyLatentImage", "inputs": {"width": 1024, "height": 1024, "batch_size": 1}},
        "6": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt, "clip": ["10",1]}},
        "7": {"class_type": "CLIPTextEncode", "inputs": {"text": "people, human, model, face, text, watermark, logo, bright flash, HDR, warm tones", "clip": ["10",1]}},
        "8": {"class_type": "VAEDecode", "inputs": {"samples": ["3",0], "vae": ["10",2]}},
        "9": {"class_type": "SaveImage", "inputs": {"filename_prefix": "wing_output", "images": ["8",0]}},
        "10": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": "flux1-dev-fp8.safetensors"}}
    }
    
    r = requests.post("http://127.0.0.1:8188/api/prompt", json={"prompt": wf})
    pid = r.json()["prompt_id"]
    
    while True:
        h = requests.get(f"http://127.0.0.1:8188/history/{pid}")
        if h.status_code == 200: break
        time.sleep(2)
    
    for nid, no in h.json()[pid]["outputs"].items():
        for img in no.get("images", []):
            img_r = requests.get(f"http://127.0.0.1:8188/api/view?filename={img['filename']}&type=output")
            out = f"/content/wing_output/img_{idx+1:02d}.png"
            with open(out, "wb") as f: f.write(img_r.content)
            results.append(out)
            display(IPImage(out))

print(f"\n🎉 All {len(results)} images done!")

In [ ]:
# @title 4. Download ZIP
import zipfile
from google.colab import files
zip_path = "/content/wing_photos.zip"
with zipfile.ZipFile(zip_path, 'w') as zf:
    for img in results:
        zf.write(img, os.path.basename(img))
files.download(zip_path)
print("✅ Done!")